# Core Utilities

> Configuration, paths, download utilities, and geographic code handling for the urban economics pipeline


In [1]:
# | default_exp core

## Configuration and Paths

The project organizes data by country (UK/US) and dataset type. This section defines the directory structure and path utilities.


In [2]:
# | export
import time
from pathlib import Path
from typing import Any, Literal

import requests

In [3]:
# | export
# Country type
Country = Literal["uk", "us"]

In [4]:
# | export
# Project root - navigate up from src/core.py location
PROJECT_ROOT = Path(__file__).parent.parent if "__file__" in globals() else Path.cwd()

# Base directories
DATA_DIR = PROJECT_ROOT / "data"
PLOTS_DIR = PROJECT_ROOT / "plots"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
DOCS_DIR = PROJECT_ROOT / "docs"

# Data subdirectories
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
OUTPUT_DIR = DATA_DIR / "output"

### Path Helpers

These functions provide consistent paths for raw data, processed data, and outputs organized by country and dataset.


In [ ]:
# | export
def raw_path(
    country: Country,  # Country code ("uk" or "us")
    dataset: str,  # Dataset name (e.g., "epc", "pipr", "ashe")
) -> Path:
    """Get path to raw data directory for a country/dataset."""
    return RAW_DATA_DIR / country / dataset


def processed_path(
    country: Country,  # Country code ("uk" or "us")
    dataset: str,  # Dataset name
) -> Path:
    """Get path to processed data directory for a country/dataset."""
    return PROCESSED_DATA_DIR / country / dataset


def output_path(
    replication: str,  # Replication name (e.g., "yimby_rent_map", "cooped_up")
) -> Path:
    """Get path to output directory for a paper replication."""
    return OUTPUT_DIR / replication

Examples:

In [ ]:
# Get paths for UK EPC data
uk_epc_raw = raw_path("uk", "epc")
uk_epc_processed = processed_path("uk", "epc")
cooped_up_out = output_path("cooped_up")

# Test that paths have correct structure
assert uk_epc_raw.parts[-3:] == ("raw", "uk", "epc")
assert uk_epc_processed.parts[-3:] == ("processed", "uk", "epc")
assert cooped_up_out.parts[-2:] == ("output", "cooped_up")

In [6]:
# | export
def ensure_dirs() -> None:
    """Create all required directories if they don't exist."""
    dirs = [
        RAW_DATA_DIR,
        PROCESSED_DATA_DIR,
        OUTPUT_DIR,
        PLOTS_DIR,
        # UK raw data
        raw_path("uk", "epc"),
        raw_path("uk", "pipr"),
        raw_path("uk", "ashe"),
        raw_path("uk", "census"),
        raw_path("uk", "geography"),
        raw_path("uk", "construction"),
        # US raw data
        raw_path("us", "cbp"),
        raw_path("us", "acs"),
        raw_path("us", "saiz"),
        raw_path("us", "geography"),
        # Processed data
        processed_path("uk", "epc"),
        processed_path("uk", "pipr"),
        processed_path("us", "cbp"),
        # Outputs
        output_path("yimby_rent_map"),
        output_path("cooped_up"),
        output_path("hsieh_moretti"),
    ]
    for d in dirs:
        d.mkdir(parents=True, exist_ok=True)

# | export
def fetch_with_retry(
    url: str,  # URL to fetch
    params: dict[str, Any] | None = None,  # Optional query parameters
    max_retries: int = 3,  # Maximum number of retry attempts
    timeout: int = 120,  # Request timeout in seconds
) -> dict[str, Any]:
    """Fetch JSON data from URL with exponential backoff retry logic."""
    for attempt in range(max_retries):
        try:
            response = requests.get(url, params=params, timeout=timeout)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as e:
            if attempt < max_retries - 1:
                wait_time = 2**attempt  # Exponential backoff: 1, 2, 4 seconds
                print(f"  Request failed, retrying in {wait_time}s... ({e})")
                time.sleep(wait_time)
            else:
                raise
    return {}


def download_file(
    url: str,  # URL to download from
    output_path: Path,  # Local path to save file
    chunk_size: int = 8192,  # Download chunk size in bytes
    timeout: int = 300,  # Request timeout in seconds
) -> Path:
    """Download a file from URL to local path with streaming."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    response = requests.get(url, stream=True, timeout=timeout)
    response.raise_for_status()

    with open(output_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=chunk_size):
            f.write(chunk)

    return output_path

In [7]:
# | export
def fetch_with_retry(
    url: str,
    params: dict[str, Any] | None = None,
    max_retries: int = 3,
    timeout: int = 120,
) -> dict[str, Any]:
    """
    Fetch JSON data from URL with retry logic for transient failures.

    Args:
        url: URL to fetch
        params: Optional query parameters
        max_retries: Maximum number of retry attempts
        timeout: Request timeout in seconds

    Returns:
        Parsed JSON response

    Raises:
        requests.exceptions.RequestException: If all retries fail
    """
    for attempt in range(max_retries):
        try:
            response = requests.get(url, params=params, timeout=timeout)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as e:
            if attempt < max_retries - 1:
                wait_time = 2**attempt  # Exponential backoff: 1, 2, 4 seconds
                print(f"  Request failed, retrying in {wait_time}s... ({e})")
                time.sleep(wait_time)
            else:
                raise
    return {}


def download_file(
    url: str,
    output_path: Path,
    chunk_size: int = 8192,
    timeout: int = 300,
) -> Path:
    """
    Download a file from URL to local path.

    Args:
        url: URL to download from
        output_path: Local path to save file
        chunk_size: Download chunk size in bytes
        timeout: Request timeout in seconds

    Returns:
        Path to downloaded file

    Raises:
        requests.exceptions.RequestException: If download fails
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    response = requests.get(url, stream=True, timeout=timeout)
    response.raise_for_status()

    with open(output_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=chunk_size):
            f.write(chunk)

    return output_path

# | export
# UK Local Authority code prefixes
UK_LA_PREFIXES = {
    "E06": "Unitary Authority (England)",
    "E07": "Non-metropolitan District",
    "E08": "Metropolitan District",
    "E09": "London Borough",
    "W06": "Unitary Authority (Wales)",
    "S12": "Council Area (Scotland)",
    "N09": "Local Government District (NI)",
}


def is_uk_local_authority(
    code: str,  # Geographic code to check
) -> bool:
    """Check if a code is a UK local authority code."""
    if not code or len(code) < 3:
        return False
    prefix = code[:3]
    return prefix in UK_LA_PREFIXES


def is_england_wales_la(
    code: str,  # Geographic code to check
) -> bool:
    """Check if a code is an England or Wales local authority."""
    if not code or len(code) < 3:
        return False
    prefix = code[:3]
    return prefix in ("E06", "E07", "E08", "E09", "W06")


def get_la_type(
    code: str,  # LA code
) -> str | None:
    """Get the type of local authority from its code."""
    if not code or len(code) < 3:
        return None
    return UK_LA_PREFIXES.get(code[:3])


# Known LA code corrections (ONS data sometimes has typos)
UK_LA_CODE_FIXES = {
    "E08000039": "E08000019",  # Sheffield (typo in ONS data)
    "E08000038": "E08000016",  # Barnsley (typo in ONS data)
}


def fix_la_code(
    code: str,  # LA code to fix
) -> str:
    """Apply known LA code corrections."""
    return UK_LA_CODE_FIXES.get(code, code)

In [8]:
# | export
# UK Local Authority code prefixes
UK_LA_PREFIXES = {
    "E06": "Unitary Authority (England)",
    "E07": "Non-metropolitan District",
    "E08": "Metropolitan District",
    "E09": "London Borough",
    "W06": "Unitary Authority (Wales)",
    "S12": "Council Area (Scotland)",
    "N09": "Local Government District (NI)",
}


def is_uk_local_authority(code: str) -> bool:
    """
    Check if a code is a UK local authority code.

    Args:
        code: Geographic code to check

    Returns:
        True if code is a UK LA code
    """
    if not code or len(code) < 3:
        return False
    prefix = code[:3]
    return prefix in UK_LA_PREFIXES


def is_england_wales_la(code: str) -> bool:
    """
    Check if a code is an England or Wales local authority.

    Args:
        code: Geographic code to check

    Returns:
        True if code is an England/Wales LA code
    """
    if not code or len(code) < 3:
        return False
    prefix = code[:3]
    return prefix in ("E06", "E07", "E08", "E09", "W06")


def get_la_type(code: str) -> str | None:
    """
    Get the type of local authority from its code.

    Args:
        code: LA code

    Returns:
        LA type description or None if not recognised
    """
    if not code or len(code) < 3:
        return None
    return UK_LA_PREFIXES.get(code[:3])


# Known LA code corrections (ONS data sometimes has typos)
UK_LA_CODE_FIXES = {
    "E08000039": "E08000019",  # Sheffield (typo in ONS data)
    "E08000038": "E08000016",  # Barnsley (typo in ONS data)
}


def fix_la_code(code: str) -> str:
    """
    Apply known LA code corrections.

    Args:
        code: LA code to fix

    Returns:
        Corrected LA code
    """
    return UK_LA_CODE_FIXES.get(code, code)

## Tests


In [9]:
# | hide
# Test path functions
assert raw_path("uk", "epc").parts[-3:] == ("raw", "uk", "epc")
assert processed_path("us", "cbp").parts[-3:] == ("processed", "us", "cbp")
assert output_path("cooped_up").parts[-2:] == ("output", "cooped_up")

# Test geographic code functions
assert is_uk_local_authority("E06000001")
assert not is_uk_local_authority("Z99999999")
assert is_england_wales_la("E09000001")  # London
assert not is_england_wales_la("S12000033")  # Scotland
assert get_la_type("E08000001") == "Metropolitan District"
assert fix_la_code("E08000039") == "E08000019"  # Sheffield fix
assert fix_la_code("E06000001") == "E06000001"  # No fix needed

## Example Usage


In [10]:
# Show project structure
print(f"Project root: {PROJECT_ROOT}")
print(f"UK EPC raw path: {raw_path('uk', 'epc')}")
print(f"UK EPC processed path: {processed_path('uk', 'epc')}")
print(f"Cooped Up output path: {output_path('cooped_up')}")

Project root: /Users/henrydashwood/Documents/housing_prices/nbs
UK EPC raw path: /Users/henrydashwood/Documents/housing_prices/nbs/data/raw/uk/epc
UK EPC processed path: /Users/henrydashwood/Documents/housing_prices/nbs/data/processed/uk/epc
Cooped Up output path: /Users/henrydashwood/Documents/housing_prices/nbs/data/output/cooped_up


In [11]:
# Demo geographic code functions
test_codes = ["E09000001", "E08000019", "W06000001", "S12000033"]
for code in test_codes:
    print(f"{code}: {get_la_type(code)}")

E09000001: London Borough
E08000019: Metropolitan District
W06000001: Unitary Authority (Wales)
S12000033: Council Area (Scotland)


In [12]:
# | hide
import nbdev

nbdev.nbdev_export()